In [8]:
# VS Code에서 실행 가능한 전체 학습 파이프라인 구성
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments
from transformers import DataCollatorForTokenClassification

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 및 토크나이저 로드
model_name = "klue/roberta-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=4).to(device)

# 데이터셋 로드
ds = load_dataset("nayohan/KOLD", cache_dir="./dataset_cache")

# 전처리 함수
def encode_example(example):
    text = example["comment"]

    def safe_int_list(raw):
        return [int(i) for i in raw if isinstance(i, (int, float)) or (isinstance(i, str) and i.strip().isdigit())]

    off_span = safe_int_list(example["OFF_span"])
    tgt_span = safe_int_list(example["TGT_span"])

    tokens = tokenizer(text, truncation=True, padding='max_length', max_length=128, return_offsets_mapping=True)
    labels = [0] * len(tokens["input_ids"])

    for idx in off_span:
        if idx < len(labels):
            labels[idx] = 1  # offensive

    for idx in tgt_span:
        if idx < len(labels):
            if labels[idx] == 1:
                labels[idx] = 3  # offensive + target
            elif labels[idx] == 0:
                labels[idx] = 2  # target

    tokens["labels"] = labels
    tokens.pop("offset_mapping")  # Trainer가 사용하지 않으므로 제거
    return tokens

# 데이터셋에 전처리 적용
encoded_ds = ds.map(encode_example)

# 데이터 collator 설정
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
)

# Trainer 정의
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_ds["train"],
    eval_dataset=encoded_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 학습 시작
trainer.train()

# 모델 저장
trainer.save_model("./saved_model")
tokenizer.save_pretrained("./saved_model")


AttributeError: partially initialized module 'datasets' has no attribute 'utils' (most likely due to a circular import)